# Example 16 — Wideband Direction-Finding Dataset

**Level:** Intermediate–Advanced

After working through this notebook you will know how to:

- Configure a `WidebandDirectionFindingDataset` with multiple co-channel sources at different center frequencies
- Understand `WidebandDFTarget` (per-signal azimuth, frequency, SNR, label)
- Visualise multi-antenna wideband spectrograms
- Apply sub-band frequency downconversion to isolate each source
- Use MUSIC on the narrowband sub-band captures for per-source DoA

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt
from plot_helpers import savefig
from spectra.algorithms import find_peaks_doa, music
from spectra.arrays import ula
from spectra.datasets import WidebandDirectionFindingDataset
from spectra.waveforms import BPSK, QAM16, QPSK

FREQ_HZ      = 2.4e9
N_ELEMENTS   = 8
SPACING      = 0.5
N_SNAPSHOTS  = 512
SAMPLE_RATE  = 20e6
CAPTURE_BW   = 16e6
N_SOURCES    = 3
SNR_RANGE    = (15.0, 25.0)
N_SAMPLES    = 100
SEED         = 42
MIN_FREQ_SEP = 3e6
MIN_ANG_SEP  = np.deg2rad(15)

SCAN_DEG = np.linspace(5, 175, 512)
SCAN_RAD = np.deg2rad(SCAN_DEG)

## 1. Dataset Configuration

In `DirectionFindingDataset` all sources share the same carrier frequency (the array's
reference frequency). In `WidebandDirectionFindingDataset` each source occupies a
distinct sub-band within `capture_bandwidth`. The frequency-dependent steering vector
rescales element positions to wavelengths at each source's carrier, so
spatially-coloured interference from co-frequency signals is physically accurate.
The `min_freq_separation` parameter enforces a guard band between sources.

In [ ]:
arr = ula(num_elements=N_ELEMENTS, spacing=SPACING, frequency=FREQ_HZ)
signal_pool = [
    BPSK(samples_per_symbol=2),
    QPSK(samples_per_symbol=2),
    QAM16(samples_per_symbol=2),
]

ds = WidebandDirectionFindingDataset(
    array=arr,
    signal_pool=signal_pool,
    num_signals=N_SOURCES,
    num_snapshots=N_SNAPSHOTS,
    sample_rate=SAMPLE_RATE,
    capture_bandwidth=CAPTURE_BW,
    snr_range=SNR_RANGE,
    azimuth_range=(np.deg2rad(10), np.deg2rad(170)),
    elevation_range=(0.0, 0.0),
    min_freq_separation=MIN_FREQ_SEP,
    min_angular_separation=MIN_ANG_SEP,
    num_samples=N_SAMPLES,
    seed=SEED,
)

data, target = ds[0]
print(f"Dataset length: {len(ds)}")
print("Sample 0 ground truth:")
for k in range(target.num_signals):
    print(f"  Source {k}: az={np.rad2deg(target.azimuths[k]):.1f}°, "
          f"f_c={target.center_freqs[k]/1e6:+.1f} MHz, "
          f"SNR={target.snrs[k]:.1f} dB, label={target.labels[k]}")

## 2. Wideband Spectrogram

The Short-Time Fourier Transform (STFT) of the element-0 IQ reveals the frequency
content of the wideband capture.  Each source appears as a horizontal band at
its center frequency.  Dashed red lines mark the true center frequencies from
`WidebandDFTarget.center_freqs`.

In [ ]:
iq0 = data[0, 0, :].numpy() + 1j * data[0, 1, :].numpy()
NFFT = 128
hop = 32
freqs_stft = np.fft.fftshift(np.fft.fftfreq(NFFT, d=1.0 / SAMPLE_RATE))
n_frames = (N_SNAPSHOTS - NFFT) // hop + 1
spec = np.zeros((NFFT, n_frames), dtype=complex)
for fi in range(n_frames):
    seg = iq0[fi * hop: fi * hop + NFFT]
    spec[:, fi] = np.fft.fftshift(np.fft.fft(seg * np.hanning(NFFT)))
spec_db = 10 * np.log10(np.abs(spec) ** 2 + 1e-30)

fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(
    spec_db, aspect="auto", origin="lower",
    extent=[0, n_frames, freqs_stft[0] / 1e6, freqs_stft[-1] / 1e6],
    cmap="viridis",
)
for fc in target.center_freqs:
    ax.axhline(fc / 1e6, color="crimson", linestyle="--", linewidth=1.2)
ax.set_xlabel("Frame")
ax.set_ylabel("Frequency (MHz)")
ax.set_title("Wideband Spectrogram — element 0 (sample 0)")
plt.tight_layout()
savefig("16_wideband_spectrogram.png")
plt.show()

## 3. Sub-Band MUSIC

Standard narrowband MUSIC assumes all sources share one carrier. For wideband
captures we isolate each source by:
1. Frequency-shifting the wideband snapshot to put source *k* at DC
2. Low-pass filtering to keep a narrow sub-band
3. Applying MUSIC to the filtered sub-band

This approximation is accurate when the source bandwidth is small relative to
the array's spatial resolution.

In [ ]:
X_full = data[:, 0, :].numpy() + 1j * data[:, 1, :].numpy()  # (N, T)

fig, axes = plt.subplots(N_SOURCES, 1, figsize=(9, 3 * N_SOURCES), sharex=True)
for k, ax in enumerate(axes):
    fc = target.center_freqs[k]
    true_az = target.azimuths[k]

    t = np.arange(N_SNAPSHOTS) / SAMPLE_RATE
    X_shift = X_full * np.exp(-1j * 2 * np.pi * fc * t)[np.newaxis, :]
    BW_sub = 2e6
    fft_len = N_SNAPSHOTS
    sub_bins = int(BW_sub / SAMPLE_RATE * fft_len)
    X_fft = np.fft.fft(X_shift, axis=1)
    X_fft[:, sub_bins // 2: fft_len - sub_bins // 2] = 0
    X_sub = np.fft.ifft(X_fft, axis=1)

    spectrum = music(X_sub, num_sources=1, array=arr, scan_angles=SCAN_RAD)
    peaks = find_peaks_doa(spectrum, SCAN_RAD, num_peaks=1)

    ax.plot(SCAN_DEG, 10 * np.log10(spectrum / spectrum.max()), color="steelblue", linewidth=1.0)
    ax.axvline(np.rad2deg(true_az), color="crimson", linestyle="--", linewidth=1.5,
               label=f"True {np.rad2deg(true_az):.1f}°")
    ax.axvline(np.rad2deg(peaks[0]), color="orange", linestyle=":", linewidth=1.5,
               label=f"Est. {np.rad2deg(peaks[0]):.1f}°")
    ax.set_ylabel(f"Source {k}\n{fc/1e6:+.1f} MHz")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Azimuth (degrees)")
fig.suptitle("Sub-Band MUSIC per Source")
plt.tight_layout()
savefig("16_subband_music.png")
plt.show()

## 4. Frequency Distribution

The dataset draws center frequencies uniformly from `(-capture_bandwidth/2, +capture_bandwidth/2)`
subject to the `min_freq_separation` constraint.  The histogram below shows the marginal
distribution over all samples.

In [ ]:
all_freqs_mhz = []
for i in range(len(ds)):
    _, t = ds[i]
    all_freqs_mhz.extend([f / 1e6 for f in t.center_freqs])

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(all_freqs_mhz, bins=40, color="steelblue", edgecolor="white")
ax.set_xlabel("Center frequency (MHz relative to DC)")
ax.set_ylabel("Count")
ax.set_title(f"Source frequency distribution ({N_SAMPLES} samples × {N_SOURCES} sources)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
savefig("16_freq_distribution.png")
plt.show()